# Phase 6 Validation Harness + Leakage-Safe Preprocessing

This notebook implements Phase 6 only for the Reto Tokio / GCI World NFL Draft Prediction project. It creates a reproducible validation harness, OOF predictions, slice diagnostics, and a validation report. It does not generate submissions, run HPO, compare model families, use public leaderboard feedback, or implement Phase 7 feature blocks.

## Scope and frozen decisions

- Metric: ROC-AUC from positive-class probabilities for `Drafted = 1`.
- Probability extraction verifies `estimator.classes_`.
- CV: `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`.
- Feature matrix excludes `Id`, `Drafted`, and `School`.
- `School`, `Age_missing`, measurement completeness, and rare-school grouping are diagnostic-only.
- `logs/experiment_log.csv` is intentionally left unchanged; a candidate row is written under `outputs/reports/`.

In [1]:
from __future__ import annotations
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "input" / "train.csv").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root containing data/input/train.csv")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
PROJECT_SEED = 42
EXPERIMENT_ID = "phase6_rf_sanity_baseline_v1"
RARE_SCHOOL_THRESHOLD = 5
N_SPLITS = 5
TARGET_COL = "Drafted"
ID_COL = "Id"
ROLE_CATEGORICAL_COLUMNS = ["Player_Type", "Position_Type", "Position"]
PHYSICAL_TEST_COLUMNS = ["Sprint_40yd", "Vertical_Jump", "Bench_Press_Reps", "Broad_Jump", "Agility_3cone", "Shuttle"]
EXCLUDED_FEATURE_COLUMNS = [ID_COL, TARGET_COL, "School"]

DATA_DIR = PROJECT_ROOT / "data" / "input"
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks" / "03_validation_harness_phase6.ipynb"
FOLD_DIR = PROJECT_ROOT / "outputs" / "folds"
OOF_DIR = PROJECT_ROOT / "outputs" / "oof"
VALIDATION_DIR = PROJECT_ROOT / "outputs" / "validation"
REPORT_DIR = PROJECT_ROOT / "outputs" / "reports"
FOLD_ASSIGNMENT_PATH = FOLD_DIR / f"{EXPERIMENT_ID}_fold_assignments.csv"
OOF_PREDICTION_PATH = OOF_DIR / f"{EXPERIMENT_ID}_oof_predictions.csv"
SLICE_REPORT_PATH = VALIDATION_DIR / f"{EXPERIMENT_ID}_slice_report.csv"
VALIDATION_REPORT_PATH = REPORT_DIR / f"{EXPERIMENT_ID}_validation_report.md"
LOG_CANDIDATE_PATH = REPORT_DIR / f"{EXPERIMENT_ID}_experiment_log_candidate.csv"
MAIN_EXPERIMENT_LOG_PATH = PROJECT_ROOT / "logs" / "experiment_log.csv"
EXPECTED_ARTIFACT_PATHS = [FOLD_ASSIGNMENT_PATH, OOF_PREDICTION_PATH, SLICE_REPORT_PATH, VALIDATION_REPORT_PATH, LOG_CANDIDATE_PATH]

print(f"Project root: {PROJECT_ROOT}")
print(f"Experiment ID: {EXPERIMENT_ID}")
print(f"Project seed: {PROJECT_SEED}")

existing = [str(p.relative_to(PROJECT_ROOT)) for p in EXPECTED_ARTIFACT_PATHS if p.exists()]
if existing:
    raise FileExistsError("Refusing to overwrite existing Phase 6 artifacts: " + "; ".join(existing))

missing_files = [str(p.relative_to(PROJECT_ROOT)) for p in [TRAIN_PATH, TEST_PATH, SAMPLE_SUBMISSION_PATH] if not p.exists()]
if missing_files:
    raise FileNotFoundError(f"Missing official files: {missing_files}")
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

checks = []
def add_check(name, passed, notes=""):
    checks.append({"check": name, "passed": bool(passed), "notes": notes})

add_check("train_csv_exists", TRAIN_PATH.exists(), str(TRAIN_PATH.relative_to(PROJECT_ROOT)))
add_check("test_csv_exists", TEST_PATH.exists(), str(TEST_PATH.relative_to(PROJECT_ROOT)))
add_check("sample_submission_csv_exists", SAMPLE_SUBMISSION_PATH.exists(), str(SAMPLE_SUBMISSION_PATH.relative_to(PROJECT_ROOT)))
add_check("target_exists_in_train", TARGET_COL in train.columns)
add_check("target_absent_from_test", TARGET_COL not in test.columns)
add_check("id_exists_in_train", ID_COL in train.columns)
add_check("id_exists_in_test", ID_COL in test.columns)
add_check("id_exists_in_sample_submission", ID_COL in sample_submission.columns)
add_check("sample_submission_columns", list(sample_submission.columns) == [ID_COL, TARGET_COL], str(list(sample_submission.columns)))
add_check("test_rows_match_sample_submission", len(test) == len(sample_submission), f"test={len(test)} sample={len(sample_submission)}")
add_check("test_ids_match_sample_submission_order", test[ID_COL].reset_index(drop=True).equals(sample_submission[ID_COL].reset_index(drop=True)) if ID_COL in test and ID_COL in sample_submission else False)
add_check("train_duplicate_full_rows", train.duplicated().sum() == 0, f"duplicates={int(train.duplicated().sum())}")
add_check("test_duplicate_full_rows", test.duplicated().sum() == 0, f"duplicates={int(test.duplicated().sum())}")
if TARGET_COL in train.columns:
    target_values = sorted(pd.Series(train[TARGET_COL].dropna().unique()).tolist())
    add_check("target_is_binary", set(target_values).issubset({0, 1, 0.0, 1.0}) and len(target_values) == 2, f"values={target_values}")
    add_check("positive_class_exists", 1 in set(target_values) or 1.0 in set(target_values), f"values={target_values}")
    add_check("test_columns_match_train_without_target", list(test.columns) == [c for c in train.columns if c != TARGET_COL])
else:
    add_check("target_is_binary", False, "target missing")
    add_check("positive_class_exists", False, "target missing")
contract_df = pd.DataFrame(checks)
failed_checks = contract_df[~contract_df["passed"]]
if not failed_checks.empty:
    raise ValueError("Data contract failed:\n" + failed_checks.to_string(index=False))
feature_candidates = [c for c in train.columns if c not in EXCLUDED_FEATURE_COLUMNS]
numeric_columns = [c for c in feature_candidates if pd.api.types.is_numeric_dtype(train[c])]
categorical_columns = [c for c in ROLE_CATEGORICAL_COLUMNS if c in feature_candidates]
feature_columns = numeric_columns + categorical_columns
for forbidden in EXCLUDED_FEATURE_COLUMNS:
    if forbidden in feature_columns:
        raise ValueError(f"Forbidden feature leaked into feature matrix: {forbidden}")

# Diagnostic-only columns. These must never enter the feature matrix, preprocessing list, fold logic, model selection, HPO, or submission generation.
diagnostics = train[[ID_COL, TARGET_COL]].copy()
for col in ["Player_Type", "Position_Type", "Year"]:
    if col in train.columns:
        diagnostics[col] = train[col]

diagnostic_notes = {}
if "Age" in train.columns:
    diagnostics["Age_missing"] = np.where(train["Age"].isna(), "missing", "present")
    diagnostic_notes["Age_missing"] = "computed from Age.isna(); diagnostic-only"
else:
    diagnostics["Age_missing"] = "not_applicable"
    diagnostic_notes["Age_missing"] = "Age column missing; affected slice marked not_applicable"

present_physical = [c for c in PHYSICAL_TEST_COLUMNS if c in train.columns]
missing_physical = [c for c in PHYSICAL_TEST_COLUMNS if c not in train.columns]
diagnostic_notes["missing_physical_test_columns"] = missing_physical
if present_physical:
    diagnostics["physical_missing_count"] = train[present_physical].isna().sum(axis=1)
    diagnostics["available_measurement_count"] = train[present_physical].notna().sum(axis=1)
    max_available = len(present_physical)
    diagnostics["measurement_completeness_group"] = np.select(
        [diagnostics["available_measurement_count"] == 0, diagnostics["available_measurement_count"] == max_available],
        ["none", "complete"],
        default="partial",
    )
else:
    diagnostics["physical_missing_count"] = np.nan
    diagnostics["available_measurement_count"] = np.nan
    diagnostics["measurement_completeness_group"] = "not_applicable"
    diagnostic_notes["measurement_completeness_group"] = "No physical-test columns present; affected slices marked not_applicable"

if "School" in train.columns:
    school_key = train["School"].fillna("__MISSING_SCHOOL__")
    school_counts = school_key.value_counts(dropna=False)
    row_counts = school_key.map(school_counts)
    diagnostics["frequent_vs_rare_school_group"] = np.where(row_counts < RARE_SCHOOL_THRESHOLD, "rare", "frequent")
    diagnostics.loc[school_key == "__MISSING_SCHOOL__", "frequent_vs_rare_school_group"] = "missing_school"
    diagnostic_notes["frequent_vs_rare_school_group"] = f"diagnostic-only train threshold: rare if School count < {RARE_SCHOOL_THRESHOLD}; frequent otherwise"
else:
    diagnostics["frequent_vs_rare_school_group"] = "not_applicable"
    diagnostic_notes["frequent_vs_rare_school_group"] = "School column missing; affected slice marked not_applicable"

for col in ["Age_missing", "physical_missing_count", "available_measurement_count", "measurement_completeness_group", "frequent_vs_rare_school_group"]:
    if col in feature_columns:
        raise ValueError(f"Diagnostic-only column leaked into feature matrix: {col}")

try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

transformers = []
if numeric_columns:
    transformers.append(("numeric", SimpleImputer(strategy="median"), numeric_columns))
if categorical_columns:
    cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", encoder)])
    transformers.append(("categorical", cat_pipe, categorical_columns))
if not transformers:
    raise ValueError("No usable features after Phase 6 exclusions")

X = train[feature_columns].copy()
y = train[TARGET_COL].astype(int).copy()
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=PROJECT_SEED)
fold_assignments = pd.DataFrame({ID_COL: train[ID_COL].to_numpy(), "fold": -1})
for fold, (_, valid_idx) in enumerate(skf.split(X, y)):
    fold_assignments.loc[valid_idx, "fold"] = fold
if (fold_assignments["fold"] < 0).any():
    raise ValueError("At least one training row did not receive a fold")
if sorted(fold_assignments["fold"].unique().tolist()) != list(range(N_SPLITS)):
    raise ValueError("Unexpected fold labels")

fold_balance = train[[ID_COL, TARGET_COL]].merge(fold_assignments, on=ID_COL, how="left", validate="one_to_one")
fold_balance = fold_balance.groupby("fold", as_index=False).agg(n=(TARGET_COL, "size"), n_positive=(TARGET_COL, "sum"), positive_rate=(TARGET_COL, "mean"))
fold_balance["n_negative"] = fold_balance["n"] - fold_balance["n_positive"]
fold_balance = fold_balance[["fold", "n", "n_positive", "n_negative", "positive_rate"]]

fold_rows = []
oof_parts = []
fold_array = fold_assignments["fold"].to_numpy()

def get_positive_class_proba(estimator, X_valid, positive_label=1):
    model = estimator.named_steps["model"]
    classes = getattr(model, "classes_", None)
    if classes is None:
        raise ValueError("Estimator classes_ not available after fit")
    matches = np.where(classes == positive_label)[0]
    if len(matches) != 1:
        raise ValueError(f"Positive label {positive_label} not found exactly once in classes_: {classes}")
    proba = estimator.predict_proba(X_valid)[:, int(matches[0])]
    if np.isnan(proba).any() or not np.isfinite(proba).all() or ((proba < 0) | (proba > 1)).any():
        raise ValueError("Invalid probability values detected")
    return proba

for fold in range(N_SPLITS):
    train_mask = fold_array != fold
    valid_mask = fold_array == fold
    pipe = Pipeline([
        ("preprocessor", ColumnTransformer(transformers=transformers, remainder="drop")),
        ("model", RandomForestClassifier(n_estimators=100, max_depth=5, random_state=PROJECT_SEED, n_jobs=-1)),
    ])
    pipe.fit(X.loc[train_mask], y.loc[train_mask])
    proba = get_positive_class_proba(pipe, X.loc[valid_mask], positive_label=1)
    auc = float(roc_auc_score(y.loc[valid_mask], proba))
    fold_rows.append({"fold": fold, "roc_auc": auc, "n": int(valid_mask.sum()), "n_positive": int(y.loc[valid_mask].sum()), "n_negative": int(valid_mask.sum() - y.loc[valid_mask].sum()), "positive_rate": float(y.loc[valid_mask].mean()), "status": "computed"})
    oof_parts.append(pd.DataFrame({ID_COL: train.loc[valid_mask, ID_COL].to_numpy(), "fold": fold, "y_true": y.loc[valid_mask].to_numpy(), "y_pred_proba": proba}))

fold_metrics = pd.DataFrame(fold_rows)
oof = pd.concat(oof_parts, ignore_index=True).sort_values(ID_COL).reset_index(drop=True)
if len(oof) != len(train) or oof[ID_COL].nunique() != len(train):
    raise ValueError("OOF predictions must contain exactly one row per train Id")
if oof["y_pred_proba"].isna().any() or not np.isfinite(oof["y_pred_proba"]).all() or ((oof["y_pred_proba"] < 0) | (oof["y_pred_proba"] > 1)).any():
    raise ValueError("OOF predictions failed probability checks")
mean_fold_auc = float(fold_metrics["roc_auc"].mean())
std_fold_auc = float(fold_metrics["roc_auc"].std(ddof=1))
oof_auc = float(roc_auc_score(oof["y_true"], oof["y_pred_proba"]))
print(fold_metrics)
print(f"Mean fold ROC-AUC: {mean_fold_auc:.6f}")
print(f"Std fold ROC-AUC: {std_fold_auc:.6f}")
print(f"OOF ROC-AUC: {oof_auc:.6f}")
fold_strategy = f"StratifiedKFold(n_splits={N_SPLITS}, shuffle=True, random_state={PROJECT_SEED})"
merged = oof.merge(diagnostics.drop(columns=[TARGET_COL]), on=ID_COL, how="left", validate="one_to_one")
slice_columns = ["Player_Type", "Position_Type", "Year", "Age_missing", "measurement_completeness_group", "available_measurement_count", "frequent_vs_rare_school_group"]
slice_rows = []
for slice_name in slice_columns:
    if slice_name not in merged.columns:
        slice_rows.append({"experiment_id": EXPERIMENT_ID, "fold_strategy": fold_strategy, "slice_name": slice_name, "slice_value": "not_applicable", "n": 0, "n_positive": 0, "n_negative": 0, "positive_rate": np.nan, "roc_auc": np.nan, "status": "not_applicable", "reason_if_skipped": f"{slice_name} unavailable", "notes": "Diagnostic slice unavailable"})
        continue
    for value, group in merged.groupby(slice_name, dropna=False):
        n = int(len(group))
        n_positive = int(group["y_true"].sum())
        n_negative = int(n - n_positive)
        positive_rate = float(group["y_true"].mean()) if n else np.nan
        if n == 0:
            status, reason, auc = "not_applicable", "empty slice", np.nan
        elif group["y_true"].nunique() < 2:
            status, reason, auc = "skipped_one_class", "ROC-AUC undefined because slice has only one class", np.nan
        else:
            status, reason, auc = "computed", "", float(roc_auc_score(group["y_true"], group["y_pred_proba"]))
        notes = ""
        if slice_name == "frequent_vs_rare_school_group":
            notes = f"Diagnostic-only; rare threshold < {RARE_SCHOOL_THRESHOLD} train rows; not a School encoding."
        elif slice_name in {"Age_missing", "measurement_completeness_group", "available_measurement_count"}:
            notes = "Diagnostic-only; not used as model feature."
        slice_rows.append({"experiment_id": EXPERIMENT_ID, "fold_strategy": fold_strategy, "slice_name": slice_name, "slice_value": str(value), "n": n, "n_positive": n_positive, "n_negative": n_negative, "positive_rate": positive_rate, "roc_auc": auc, "status": status, "reason_if_skipped": reason, "notes": notes})
slice_report = pd.DataFrame(slice_rows)
allowed_statuses = {"computed", "skipped_too_small", "skipped_one_class", "not_applicable"}
if not set(slice_report["status"]).issubset(allowed_statuses):
    raise ValueError("Slice report contains unsupported status")
special = slice_report[(slice_report["slice_name"] == "Player_Type") & (slice_report["slice_value"].str.lower() == "special_teams")]
special_teams_status = "not_applicable" if special.empty else str(special.iloc[0]["status"])
special_teams_notes = "No special_teams slice found" if special.empty else (str(special.iloc[0]["reason_if_skipped"]) or "computed")

for d in [FOLD_DIR, OOF_DIR, VALIDATION_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)
fold_assignments.to_csv(FOLD_ASSIGNMENT_PATH, index=False)
oof.to_csv(OOF_PREDICTION_PATH, index=False)
slice_report.to_csv(SLICE_REPORT_PATH, index=False)

log_candidate = pd.DataFrame([{
    "experiment_id": EXPERIMENT_ID,
    "date": datetime.now().strftime("%Y-%m-%d"),
    "phase": "Phase 6",
    "notebook_or_script": str(NOTEBOOK_PATH.relative_to(PROJECT_ROOT)),
    "git_commit_or_status": "review_git_status_for_exact_state",
    "data_version": "official_data_input_csvs",
    "fold_strategy": fold_strategy,
    "random_seed": PROJECT_SEED,
    "feature_block": "raw_no_school_sanity_baseline",
    "preprocessing_summary": "numeric median imputation; role categorical most_frequent imputation plus one-hot; all fitted inside folds",
    "model_family": "RandomForestClassifier",
    "model_params_summary": f"n_estimators=100; max_depth=5; random_state={PROJECT_SEED}; n_jobs=-1",
    "hpo_status": "not_run",
    "cv_auc_mean": mean_fold_auc,
    "cv_auc_std": std_fold_auc,
    "fold_scores": "; ".join(f"fold_{int(r.fold)}={r.roc_auc:.6f}" for r in fold_metrics.itertuples()),
    "slice_metrics_available": True,
    "leakage_checks_passed": True,
    "submission_created": False,
    "submission_path": None,
    "public_lb_score_if_submitted": None,
    "notes": "Phase 6 validation harness sanity baseline; no School feature; diagnostics only; no submission; no HPO",
    "decision": "validation_harness_sanity_check",
}])
log_candidate.to_csv(LOG_CANDIDATE_PATH, index=False)

leakage_checks = pd.DataFrame([
    {"check": "Feature matrix excludes Id", "passed": ID_COL not in feature_columns, "evidence": str(feature_columns)},
    {"check": "Feature matrix excludes Drafted", "passed": TARGET_COL not in feature_columns, "evidence": str(feature_columns)},
    {"check": "Feature matrix excludes School", "passed": "School" not in feature_columns, "evidence": str(feature_columns)},
    {"check": "Preprocessing fitted inside each fold", "passed": True, "evidence": "Pipeline constructed and fitted inside CV loop"},
    {"check": "Positive class probability verified via estimator.classes_", "passed": True, "evidence": "get_positive_class_proba helper"},
    {"check": "No test data fitting/tuning/selection", "passed": True, "evidence": "test used only in contract checks"},
    {"check": "No submissions generated", "passed": True, "evidence": "no write to outputs/submissions"},
    {"check": "No HPO run", "passed": True, "evidence": "single fixed RandomForestClassifier"},
    {"check": "Main experiment log unchanged by notebook", "passed": True, "evidence": "candidate log written separately"},
])

def table(df: pd.DataFrame) -> str:
    if df.empty:
        return "_No rows._"
    work = df.copy()
    columns = [str(c) for c in work.columns]

    def format_value(value) -> str:
        if pd.isna(value):
            return ""
        return str(value).replace("\n", " ").replace("|", "\\|")

    lines = ["| " + " | ".join(columns) + " |", "|" + "|".join(["---"] * len(columns)) + "|"]
    for _, row in work.iterrows():
        lines.append("| " + " | ".join(format_value(row[col]) for col in work.columns) + " |")
    return "\n".join(lines)

report = f"""# Phase 6 Validation Harness Report

## Purpose

Implement and verify a leakage-safe local validation harness. This is not a competitive modeling report.

## Data contract checks

- Train shape: `{train.shape}`
- Test shape: `{test.shape}`
- Sample submission shape: `{sample_submission.shape}`
- Contract passed: `True`
- Unit of observation: `Not confirmed yet`

{table(contract_df)}

## Feature set used

| Category | Columns | Notes |
|---|---|---|
| numeric_features | {', '.join(numeric_columns)} | Used as model features. |
| categorical_role_features | {', '.join(categorical_columns)} | Fold-safe encoded role categoricals. |
| excluded | {', '.join(EXCLUDED_FEATURE_COLUMNS)} | Excluded from feature matrix. |

## Diagnostic-only variables

| Variable | Purpose | Confirmed not used as feature? |
|---|---|---|
| Age_missing | Age missingness slice | True |
| physical_missing_count | Physical test missingness support | True |
| available_measurement_count | Measurement completeness slice | True |
| measurement_completeness_group | Completeness grouping slice | True |
| frequent_vs_rare_school_group | School frequency slice with threshold `< {RARE_SCHOOL_THRESHOLD}` | True |

Diagnostic notes:

```text
{diagnostic_notes}
```

## Phase 3 EDA risks carried into Phase 6

- School high-cardinality/rare-category risk.
- Age_missing strong train-only association but not yet accepted as feature.
- Measurement availability as future Phase 7 hypothesis.
- special_teams as important Player_Type slice.
- Year/cohort effects as diagnostic only.
- Role-dependent physical interpretation.
- Unit of observation: Not confirmed yet.

The `special_teams` slice status is `{special_teams_status}`. Notes: {special_teams_notes}.

## Preprocessing summary

- Numeric features: `SimpleImputer(strategy="median")`, fitted inside each training fold.
- Role categorical features: `SimpleImputer(strategy="most_frequent")` plus `OneHotEncoder(handle_unknown="ignore", sparse_output=False)`, fitted inside each training fold.
- No global preprocessing was fitted before cross-validation.
- No scaling was used because the Phase 6 sanity model is a RandomForest classifier.

## Model used

- Model: `RandomForestClassifier`
- Purpose: Phase 6 validation harness sanity baseline only.
- Parameters: `n_estimators=100`, `max_depth=5`, `random_state={PROJECT_SEED}`, `n_jobs=-1`.
- No model-family comparison was performed.

## Fold strategy

- Splitter: `StratifiedKFold`
- `n_splits`: `{N_SPLITS}`
- `shuffle`: `True`
- `random_state`: `{PROJECT_SEED}`
- Fold labels: `0..4`

{table(fold_balance)}

## Fold metrics

{table(fold_metrics)}

Mean fold ROC-AUC: `{mean_fold_auc:.6f}`

Std fold ROC-AUC: `{std_fold_auc:.6f}`

## OOF metrics

OOF ROC-AUC: `{oof_auc:.6f}`

OOF rows: `{len(oof)}`

OOF predictions are finite, non-missing, and within `[0, 1]`.

## Slice diagnostics summary

{table(slice_report.groupby(['slice_name', 'status'], as_index=False).size().rename(columns={'size': 'count'}))}

Full slice diagnostics are saved to `{SLICE_REPORT_PATH.relative_to(PROJECT_ROOT)}`.

Slice diagnostics are reporting-only and must not be used for Phase 6 feature selection, model selection, preprocessing selection, HPO, or submission strategy.

## Leakage checklist result

{table(leakage_checks)}

## Test-data use statement

Test data was used only for data-contract structure checks and sample-submission alignment. It was not used for fitting, tuning, preprocessing, feature selection, model selection, HPO, or submission generation.

## Public leaderboard statement

No public leaderboard feedback was used in Phase 6.

## Experiment log handling

`logs/experiment_log.csv` was intentionally left unchanged. A candidate log row was written to `{LOG_CANDIDATE_PATH.relative_to(PROJECT_ROOT)}`. Main experiment log migration/update remains deferred.

## Known limitations

- Minimum slice size remains `Not confirmed yet`; one-class slices are skipped and small slices are reported with caution.
- This sanity baseline is not a final model and is not a model-family comparison.
- `School`, `Age_missing`, and measurement-completeness diagnostics are not model features.

## Provisional decisions remaining

- Exact feature-block acceptance threshold remains unresolved until Phase 7.
- Final School encoding strategy remains unresolved until staged ablations.
- Measurement-completeness cutoffs remain diagnostic only.

## Not confirmed yet items

- Unit of observation: Not confirmed yet.
- Need for grouped CV: Not confirmed yet.
- Minimum slice-size threshold: Not confirmed yet.

## Phase 6 acceptance status

Ready for manual review.

## Next recommended step

Review Phase 6 artifacts manually. Do not start Phase 7 until Phase 6 is accepted.

## Boundary statements

- No submissions were generated.
- No HPO was run.
- No feature blocks were tested.
- No model-family comparison was performed.
- No public leaderboard feedback was used.
- Test data was not used for fitting, tuning, preprocessing, feature selection, or model selection.
- No causal interpretation is made; results are associated with validation ranking performance only.
"""
VALIDATION_REPORT_PATH.write_text(report, encoding="utf-8")

print("Artifacts written:")
for p in [FOLD_ASSIGNMENT_PATH, OOF_PREDICTION_PATH, SLICE_REPORT_PATH, VALIDATION_REPORT_PATH, LOG_CANDIDATE_PATH]:
    print(p.relative_to(PROJECT_ROOT), p.exists())
print(f"Mean fold ROC-AUC: {mean_fold_auc:.6f}")
print(f"Std fold ROC-AUC: {std_fold_auc:.6f}")
print(f"OOF ROC-AUC: {oof_auc:.6f}")


Project root: C:\GitHub\reto_Tokio
Experiment ID: phase6_rf_sanity_baseline_v1
Project seed: 42


   fold   roc_auc    n  n_positive  n_negative  positive_rate    status
0     0  0.690076  557         361         196       0.648115  computed
1     1  0.751758  556         361         195       0.649281  computed
2     2  0.761581  556         361         195       0.649281  computed
3     3  0.704939  556         360         196       0.647482  computed
4     4  0.737911  556         360         196       0.647482  computed
Mean fold ROC-AUC: 0.729253
Std fold ROC-AUC: 0.030629
OOF ROC-AUC: 0.726616


Artifacts written:
outputs\folds\phase6_rf_sanity_baseline_v1_fold_assignments.csv True
outputs\oof\phase6_rf_sanity_baseline_v1_oof_predictions.csv True
outputs\validation\phase6_rf_sanity_baseline_v1_slice_report.csv True
outputs\reports\phase6_rf_sanity_baseline_v1_validation_report.md True
outputs\reports\phase6_rf_sanity_baseline_v1_experiment_log_candidate.csv True
Mean fold ROC-AUC: 0.729253
Std fold ROC-AUC: 0.030629
OOF ROC-AUC: 0.726616


## Manual review reminder

Phase 6 artifacts require manual review before Phase 6 can be accepted. Do not start Phase 7 from this notebook.